# Modélisation Textuelle - Word2Vec Skip Gram

## Objectif
Classification des produits Rakuten en utilisant Word2Vec avec l'architecture Skip Gram pour créer des embeddings de mots, puis agréger ces embeddings pour représenter les documents.

## Approche
1. **Word2Vec Skip Gram** : Entraînement d'un modèle Word2Vec avec architecture Skip Gram
2. **Agrégation des embeddings** : Moyenne des embeddings de mots pour représenter chaque document
3. **Classification** : Utilisation de classifieurs classiques sur les embeddings agrégés


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

# Gensim pour Word2Vec
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Pour sauvegarder
import joblib
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Bibliothèques importées")


✅ Bibliothèques importées


In [2]:
# Chargement des données
X_train = pd.read_csv('X_train_update.csv')
Y_train = pd.read_csv('Y_train_CVw08PX.csv')
X_test = pd.read_csv('X_test_update.csv')

train_data = pd.merge(X_train, Y_train, left_index=True, right_index=True)
print(f"Taille train_data : {train_data.shape}")

# Fonction de nettoyage
def nettoyer_texte(texte):
    if pd.isna(texte) or texte == 'nan':
        return ''
    texte = str(texte)
    texte = re.sub(r'<[^>]+>', '', texte)
    texte = texte.replace('&nbsp;', ' ')
    texte = texte.replace('&amp;', '&')
    texte = texte.replace('&lt;', '<')
    texte = texte.replace('&gt;', '>')
    texte = texte.replace('&quot;', '"')
    texte = texte.replace('&#39;', "'")
    texte = texte.replace('&eacute;', 'é')
    texte = texte.replace('&egrave;', 'è')
    texte = texte.replace('&ecirc;', 'ê')
    texte = texte.replace('&agrave;', 'à')
    texte = re.sub(r'\s+', ' ', texte)
    return texte.strip()

# Nettoyage
train_data['texte_complet'] = (
    train_data['designation'].apply(nettoyer_texte) + ' ' + 
    train_data['description'].apply(nettoyer_texte)
).str.strip()

X_test['texte_complet'] = (
    X_test['designation'].apply(nettoyer_texte) + ' ' + 
    X_test['description'].apply(nettoyer_texte)
).str.strip()

print("✅ Données chargées et nettoyées")


Taille train_data : (84916, 7)
✅ Données chargées et nettoyées


In [3]:
# Tokenisation des textes
def tokenize(text):
    """Tokenise le texte en mots"""
    if not text or pd.isna(text):
        return []
    # Convertir en minuscules et diviser par espaces
    tokens = text.lower().split()
    # Filtrer les tokens vides
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

# Tokeniser tous les textes
print("Tokenisation des textes...")
sentences = [tokenize(text) for text in train_data['texte_complet']]
# Filtrer les phrases vides
sentences = [s for s in sentences if len(s) > 0]

print(f"✅ {len(sentences)} phrases tokenisées")
print(f"Exemple de phrase tokenisée : {sentences[0][:10]}")


Tokenisation des textes...
✅ 84915 phrases tokenisées
Exemple de phrase tokenisée : ['olivia:', 'personalisiertes', 'notizbuch', '150', 'seiten', 'punktraster', 'ca', 'din', 'a5', 'rosen-design']


In [4]:
# Callback pour suivre l'entraînement
class EpochLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
    def on_epoch_end(self, model):
        print(f"Époque {self.epoch} terminée")
        self.epoch += 1

# Paramètres Word2Vec Skip Gram
vector_size = 100  # Dimension des embeddings
window = 5  # Taille de la fenêtre contextuelle
min_count = 2  # Fréquence minimale d'un mot
sg = 1  # 0 = CBOW, 1 = Skip-gram
workers = 4  # Nombre de threads

print("Entraînement du modèle Word2Vec Skip Gram...")
print(f"Paramètres : vector_size={vector_size}, window={window}, min_count={min_count}")

# Entraîner le modèle
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    sg=sg,  # Skip Gram
    workers=workers,
    epochs=10,
    callbacks=[EpochLogger()]
)

print(f"\n✅ Modèle Word2Vec Skip Gram entraîné")
print(f"Vocabulaire : {len(w2v_model.wv)} mots")
print(f"Exemple d'embedding pour 'produit' : {w2v_model.wv['produit'][:5] if 'produit' in w2v_model.wv else 'Mot non trouvé'}")


Entraînement du modèle Word2Vec Skip Gram...
Paramètres : vector_size=100, window=5, min_count=2
Époque 0 terminée
Époque 1 terminée
Époque 2 terminée
Époque 3 terminée
Époque 4 terminée
Époque 5 terminée
Époque 6 terminée
Époque 7 terminée
Époque 8 terminée
Époque 9 terminée

✅ Modèle Word2Vec Skip Gram entraîné
Vocabulaire : 134687 mots
Exemple d'embedding pour 'produit' : [-0.11638156  0.7274962  -0.0140133   0.26095292 -0.29547316]


In [5]:
# Fonction pour obtenir l'embedding d'un document (moyenne des embeddings des mots)
def document_to_vector(text, model, vector_size):
    """Convertit un texte en vecteur en moyennant les embeddings des mots"""
    tokens = tokenize(text)
    if len(tokens) == 0:
        return np.zeros(vector_size)
    
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) == 0:
        return np.zeros(vector_size)
    
    return np.mean(vectors, axis=0)

# Créer les embeddings pour tous les documents
print("Création des embeddings de documents...")
X_embeddings = np.array([
    document_to_vector(text, w2v_model, vector_size) 
    for text in train_data['texte_complet']
])

y = train_data['prdtypecode'].values

print(f"✅ Embeddings créés : shape {X_embeddings.shape}")

# Embeddings pour X_test
X_test_embeddings = np.array([
    document_to_vector(text, w2v_model, vector_size) 
    for text in X_test['texte_complet']
])

print(f"✅ Embeddings de test créés : shape {X_test_embeddings.shape}")


Création des embeddings de documents...
✅ Embeddings créés : shape (84916, 100)
✅ Embeddings de test créés : shape (13812, 100)


In [6]:
# Division train/validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train_split.shape[0]}, Validation : {X_val_split.shape[0]}")


Train : 67932, Validation : 16984


In [7]:
# Test de différents classifieurs
results = {}

# Régression Logistique
print("="*60)
print("Régression Logistique")
print("="*60)
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
lr.fit(X_train_split, y_train_split)
y_pred_lr = lr.predict(X_val_split)
results['Logistic Regression'] = {
    'model': lr,
    'accuracy': accuracy_score(y_val_split, y_pred_lr),
    'f1_weighted': f1_score(y_val_split, y_pred_lr, average='weighted'),
    'f1_macro': f1_score(y_val_split, y_pred_lr, average='macro')
}
print(f"Accuracy: {results['Logistic Regression']['accuracy']:.4f}")
print(f"F1 (weighted): {results['Logistic Regression']['f1_weighted']:.4f}")

# Random Forest
print("\n" + "="*60)
print("Random Forest")
print("="*60)
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_split, y_train_split)
y_pred_rf = rf.predict(X_val_split)
results['Random Forest'] = {
    'model': rf,
    'accuracy': accuracy_score(y_val_split, y_pred_rf),
    'f1_weighted': f1_score(y_val_split, y_pred_rf, average='weighted'),
    'f1_macro': f1_score(y_val_split, y_pred_rf, average='macro')
}
print(f"Accuracy: {results['Random Forest']['accuracy']:.4f}")
print(f"F1 (weighted): {results['Random Forest']['f1_weighted']:.4f}")

# SVM
print("\n" + "="*60)
print("SVM")
print("="*60)
svm = SVC(kernel='linear', class_weight='balanced', random_state=42, probability=True)
svm.fit(X_train_split, y_train_split)
y_pred_svm = svm.predict(X_val_split)
results['SVM'] = {
    'model': svm,
    'accuracy': accuracy_score(y_val_split, y_pred_svm),
    'f1_weighted': f1_score(y_val_split, y_pred_svm, average='weighted'),
    'f1_macro': f1_score(y_val_split, y_pred_svm, average='macro')
}
print(f"Accuracy: {results['SVM']['accuracy']:.4f}")
print(f"F1 (weighted): {results['SVM']['f1_weighted']:.4f}")


Régression Logistique
Accuracy: 0.7355
F1 (weighted): 0.7355

Random Forest
Accuracy: 0.7800
F1 (weighted): 0.7759

SVM
Accuracy: 0.7599
F1 (weighted): 0.7603


In [8]:
# Comparaison
comparison_df = pd.DataFrame({
    'Modèle': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'F1 (weighted)': [r['f1_weighted'] for r in results.values()],
    'F1 (macro)': [r['f1_macro'] for r in results.values()]
}).sort_values('F1 (weighted)', ascending=False)

print("\n" + "="*80)
print("COMPARAISON DES MODÈLES")
print("="*80)
print(comparison_df.to_string(index=False))

# Meilleur modèle
meilleur_nom = comparison_df.iloc[0]['Modèle']
meilleur_modele = results[meilleur_nom]['model']

# Entraîner sur toutes les données
meilleur_modele_final = type(meilleur_modele)(**meilleur_modele.get_params())
meilleur_modele_final.fit(X_embeddings, y)

# Prédictions sur test
y_test_pred = meilleur_modele_final.predict(X_test_embeddings)

# Sauvegarder
os.makedirs('models', exist_ok=True)
os.makedirs('output', exist_ok=True)

w2v_model.save('models/word2vec_skipgram.model')
joblib.dump(meilleur_modele_final, f'models/skipgram_{meilleur_nom.lower().replace(" ", "_")}.pkl')

pd.DataFrame({
    'productid': X_test['productid'],
    'prdtypecode': y_test_pred
}).to_csv('output/predictions_skipgram.csv', index=False)

print(f"\n✅ Meilleur modèle : {meilleur_nom}")
print(f"✅ Modèles et prédictions sauvegardés")



COMPARAISON DES MODÈLES
             Modèle  Accuracy  F1 (weighted)  F1 (macro)
      Random Forest  0.780028       0.775862    0.758089
                SVM  0.759892       0.760282    0.733899
Logistic Regression  0.735516       0.735499    0.704925

✅ Meilleur modèle : Random Forest
✅ Modèles et prédictions sauvegardés
